# Prediction of the Algorithm

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tabulate import tabulate
import numpy as np

DATASET_FOLDER = "../data"
TEST_FOLDER = "../tests"

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
data = pd.read_csv("../results/metrics2.csv")

columns = [
"Dataset","Algorithm","Grid Size","Total Placements",
"Residential","Utility","Score","Time (s)","Peak Memory (MB)","Parameters"
]

df = pd.DataFrame(data, columns=columns)

In [3]:
def predict_with_range(model, X, multiplier=4):

    tree_preds = np.array([tree.predict(X)[0] for tree in model.estimators_])

    mean_pred = tree_preds.mean()
    std_pred = tree_preds.std()

    lower = mean_pred - multiplier * std_pred
    upper = mean_pred + multiplier * std_pred

    return mean_pred, lower, upper

In [4]:
def extract_dataset_features(filepath):
    with open(filepath, "r") as f:
        lines = f.read().strip().split("\n")

    # First line: H, W, D, B
    H, W, D, B = map(int, lines[0].split())
    projects = lines[1:]

    i = 0
    residential_areas, residential_caps = [], []
    utility_areas, utility_services = [], set()

    residential_density, utility_density = [], []

    while i < len(projects):
        parts = projects[i].split()
        p_type, h, w, value = parts[0], int(parts[1]), int(parts[2]), int(parts[3])
        area = h * w

        footprint_lines = projects[i+1:i+1+h]
        occupied_blocks = sum(line.count("#") for line in footprint_lines)
        density = occupied_blocks / (h*w)

        if p_type == "R":
            residential_areas.append(area)
            residential_caps.append(value)
            residential_density.append(density)
        else:  # Utility
            utility_areas.append(area)
            utility_services.add(value)
            utility_density.append(density)

        i += 1 + h  # skip footprint lines

    return {
        "Dataset": os.path.splitext(os.path.basename(filepath))[0],
        "grid_height": H,
        "grid_width": W,
        "grid_area": H * W,
        "max_distance": D,
        "num_projects": B,
        "num_residential_projects": len(residential_areas),
        "num_utility_projects": len(utility_areas),
        "avg_residential_area": sum(residential_areas) / len(residential_areas) if residential_areas else 0,
        "max_residential_area": max(residential_areas) if residential_areas else 0,
        "avg_residential_capacity": sum(residential_caps) / len(residential_caps) if residential_caps else 0,
        "avg_utility_area": sum(utility_areas) / len(utility_areas) if utility_areas else 0,
        "max_utility_area": max(utility_areas) if utility_areas else 0,
        "num_service_types": len(utility_services),
        "avg_residential_density": sum(residential_density) / len(residential_density) if residential_density else 0,
        "max_residential_density": max(residential_density) if residential_density else 0,
        "avg_utility_density": sum(utility_density) / len(utility_density) if utility_density else 0,
        "max_utility_density": max(utility_density) if utility_density else 0,
    }


# Extract features for all datasets
dataset_features = []

for file in os.listdir(DATASET_FOLDER):
    path = os.path.join(DATASET_FOLDER, file)
    if os.path.isfile(path):
        dataset_features.append(extract_dataset_features(path))

features_df = pd.DataFrame(dataset_features)

# Merge with your existing df
df = df.merge(features_df, on="Dataset", how="left")

df = df[df["Dataset"] != "a_example"]

print(df.head())

        Dataset      Algorithm  Grid Size  Total Placements  Residential  \
5  b_short_walk  hill_climbing  1000x1000            103780        20886   
6  b_short_walk  hill_climbing  1000x1000            103780        20886   
7  b_short_walk  hill_climbing  1000x1000            103780        20886   
8  b_short_walk  hill_climbing  1000x1000            103780        20886   
9  b_short_walk  hill_climbing  1000x1000            103780        20886   

   Utility    Score  Time (s)  Peak Memory (MB)  \
5    82894  1807423  519.4268          220.3475   
6    82894  1807423  500.0384          220.3478   
7    82894  1807423  494.1642          220.1269   
8    82894  1807423  497.6014          220.1269   
9    82894  1807423  494.4644          220.1269   

                                          Parameters  ...  \
5  max_iterations=5000, patience=500, min_delta=2...  ...   
6  max_iterations=5000, patience=500, min_delta=2...  ...   
7  max_iterations=5000, patience=500, min_delta=2... 

## Training

In [5]:
if "Grid Size" in df.columns:
    df[['Grid_H', 'Grid_W']] = df['Grid Size'].str.split('x', expand=True).astype(int)

# One-Hot Encoding for categorical variables (Dataset e Algorithm)
df_encoded = pd.get_dummies(df, columns=["Algorithm"])

cols_to_drop = [
    "Dataset", "Score", "Grid Size", "Parameters",
    "Total Placements", "Residential", "Utility",
    "Time (s)", "Peak Memory (MB)"
]
if "Timestamp" in df_encoded.columns:
    cols_to_drop.append("Timestamp")

X = df_encoded.drop(columns=cols_to_drop, errors='ignore')
y = df_encoded["Score"]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)

results = df.loc[idx_test].copy()
results["Predicted_Score"] = pred.astype(int)
results["Original_Score"] = y_test.values

results["Error (%)"] = ((abs(results["Original_Score"] - results["Predicted_Score"]) / results["Original_Score"]) * 100).round(2).astype(str) + "%"

results_display = results[["Dataset", "Algorithm", "Original_Score", "Predicted_Score", "Error (%)"]]

print("\n" + "="*80)
print("Surrogate Model: Random Forest Regressor")
print("="*80)

print(tabulate(results_display.head(20), headers='keys', tablefmt='fancy_grid', showindex=False))

print("\nGlobal Metrics:")
print(f"-> MAE (Mean Absolute Error): {mean_absolute_error(y_test, pred):.2f}")
print(f"-> R²  (Precision of the Model): {r2_score(y_test, pred):.4f}")
print("="*80 + "\n")


Surrogate Model: Random Forest Regressor
╒════════════════════════╤═════════════════════╤══════════════════╤═══════════════════╤═════════════╕
│ Dataset                │ Algorithm           │   Original_Score │   Predicted_Score │ Error (%)   │
╞════════════════════════╪═════════════════════╪══════════════════╪═══════════════════╪═════════════╡
│ e_precise_fit          │ tabu_search         │          2251735 │           2201359 │ 2.24%       │
├────────────────────────┼─────────────────────┼──────────────────┼───────────────────┼─────────────┤
│ c_going_green          │ hill_climbing       │          1578227 │           1377679 │ 12.71%      │
├────────────────────────┼─────────────────────┼──────────────────┼───────────────────┼─────────────┤
│ f_different_footprints │ hill_climbing       │          1244780 │           2186534 │ 75.66%      │
├────────────────────────┼─────────────────────┼──────────────────┼───────────────────┼─────────────┤
│ d_wide_selection       │ tabu_search  

## Test

In [6]:
algoritmos_conhecidos = df['Algorithm'].unique()
predictions = []

print(f"Analyzing new maps in folder '{TEST_FOLDER}'...\n")

for filename in os.listdir(TEST_FOLDER):
    if filename.endswith(".in"):

        filepath = os.path.join(TEST_FOLDER, filename)

        features = extract_dataset_features(filepath)

        for algo in algoritmos_conhecidos:

            input_dict = features.copy()
            input_dict["Algorithm"] = algo

            input_data = pd.DataFrame([input_dict])

            input_encoded = pd.get_dummies(input_data, columns=['Algorithm'])
            input_encoded = input_encoded.reindex(columns=X_train.columns, fill_value=0)

            mean_pred, lower, upper = predict_with_range(model, input_encoded)

            pred_score = int(mean_pred)
            lower = int(lower)
            upper = int(upper)

            predictions.append({
                "Test File": filename,
                "Algorithm": algo,
                "Predicted Score": pred_score,
                "Expected Min": lower,
                "Expected Max": upper
            })

results_df = pd.DataFrame(predictions)
results_df = results_df.sort_values(by=["Test File", "Predicted Score"], ascending=[True, False])

# 1: Everything (All Algorithms)
print("="*80)
print("Predictions for New Maps:")
print("="*80)
print(tabulate(results_df, headers='keys', tablefmt='fancy_grid', showindex=False))
print("\n")

# 2: Best Algorithm per Map
best_algorithms_df = results_df.drop_duplicates(subset=["Test File"], keep="first")
best_algorithms_df = best_algorithms_df.rename(columns={"Algorithm": "Recommended Algorithm", "Predicted Score": "Expected Score"})

print("="*80)
print("Recommended Algorithm for Each New Map:")
print("="*80)
print(tabulate(best_algorithms_df, headers='keys', tablefmt='fancy_grid', showindex=False))
print("="*80)

Analyzing new maps in folder '../tests'...

Predictions for New Maps:
╒═══════════════════════╤═════════════════════╤═══════════════════╤════════════════╤════════════════╕
│ Test File             │ Algorithm           │   Predicted Score │   Expected Min │   Expected Max │
╞═══════════════════════╪═════════════════════╪═══════════════════╪════════════════╪════════════════╡
│ g_joindb.in           │ tabu_search         │           2222126 │        2060661 │        2383591 │
├───────────────────────┼─────────────────────┼───────────────────┼────────────────┼────────────────┤
│ g_joindb.in           │ hill_climbing       │           2099489 │        1898343 │        2300635 │
├───────────────────────┼─────────────────────┼───────────────────┼────────────────┼────────────────┤
│ g_joindb.in           │ greedy              │           2095764 │        1898195 │        2293334 │
├───────────────────────┼─────────────────────┼───────────────────┼────────────────┼────────────────┤
│ g_joindb.i

## Disclosure

In [ ]:
metrics_path = "../results/tests_metrics.csv"
real_scores_df = pd.read_csv(metrics_path)

real_scores_df["Dataset"] = real_scores_df["Dataset"].str.replace(".in","")
results_df["Test File"] = results_df["Test File"].str.replace(".in","")

comparison_df = results_df.merge(
    real_scores_df.rename(columns={"Score": "Real_Score"}),
    left_on=["Test File", "Algorithm"],
    right_on=["Dataset", "Algorithm"],
    how="left"
)

comparison_df["Inside Range"] = comparison_df.apply(
    lambda row: (
        "YES"
        if pd.notna(row["Real_Score"]) and row["Expected Min"] <= row["Real_Score"] <= row["Expected Max"]
        else "NO"
    ),
    axis=1
)

display_df = comparison_df[[
    "Test File",
    "Algorithm",
    "Real_Score",
    "Predicted Score",
    "Expected Min",
    "Expected Max",
    "Inside Range"
]].sort_values(by=["Test File"])

print("\n" + "="*80)
print("Comparison: Predicted vs Real Scores")
print("="*80)
print(tabulate(display_df, headers='keys', tablefmt='fancy_grid', showindex=False))

# Percentage of YES vs NO
total = len(display_df)
yes_count = (display_df["Inside Range"] == "YES").sum()
no_count = (display_df["Inside Range"] == "NO").sum()
print("\nSummary of Predictions vs Real Scores:")
print(f"-> Total Predictions: {total}")
print(f"-> Inside Range (YES): {yes_count} ({(yes_count/total)*100:.2f}%)")
print(f"-> Outside Range (NO): {no_count} ({(no_count/total)*100:.2f}%)")


Comparison: Predicted vs Real Scores
╒════════════════════╤═════════════════════╤══════════════╤═══════════════════╤════════════════╤════════════════╤════════════════╕
│ Test File          │ Algorithm           │   Real_Score │   Predicted Score │   Expected Min │   Expected Max │ Inside Range   │
╞════════════════════╪═════════════════════╪══════════════╪═══════════════════╪════════════════╪════════════════╪════════════════╡
│ g_joindb           │ tabu_search         │      2460870 │           2222126 │        2060661 │        2383591 │ NO             │
├────────────────────┼─────────────────────┼──────────────┼───────────────────┼────────────────┼────────────────┼────────────────┤
│ g_joindb           │ hill_climbing       │      1900573 │           2099489 │        1898343 │        2300635 │ YES            │
├────────────────────┼─────────────────────┼──────────────┼───────────────────┼────────────────┼────────────────┼────────────────┤
│ g_joindb           │ greedy              │ 

# Results